### Exploration of MWPM decoding graph and edge reweighting, similar to the paper: *DGR: Tackling Drifted and Correlated Noise in Quantum Error Correction via Decoding Graph Re-weighting*

In [17]:
import numpy as np
import stim
from surface_code_stim import SurfaceCode
import pymatching
import plotly.graph_objects as go

1. Build a circuit-level noise circuit for the surface code using Stim. We have two possibilities:
- Custom implementation of the faulty syndrome extraction circuit (from a previous project). Main drawback: currently, detectors of X stabilizers are not considered for Z memory.
- Stim built-in implementation. The noise model is not flexible to adding more sophisticated error mechanisms.

In [ ]:
# Build the circuit manually, using custom error channels, to have more control over the error model.

# Ignore the decoherence times and gates durations for now.
hardware_params = {
        'T1': 250e-6,  # in seconds
        'T2': 250e-6,  # in seconds
        'duration_1q_gate': 20e-9,  # in seconds
        'duration_2q_gate': 30e-9,  # in seconds
        'state_prep_error': 1e-2,
        'measurement_error': 1e-2,
        'gate_error_1q': 1e-2,
        'gate_error_2q': 1e-2,
        'duration_readout': 500e-9
    }

sc = SurfaceCode(
        hardware_params=hardware_params,
        distance=3,
        n_rounds=3,
        ad=False,
        crosstalk=False,
        default_2q_gate='CX', # Can select CZ gate as well
        memory_type='z', # Can select x memory as well
        idle_depol=True
    )

sc_circuit_custom = sc.build_surface_code_circuit()

# Build the circuit using stim's built-in generator, which has a fixed error model.
sc_circuit_builtin = stim.Circuit.generated("surface_code:rotated_memory_z",
                                distance=3,
                                rounds=3,
                                after_clifford_depolarization=0.01,
                                before_measure_flip_probability=0.01,
                                after_reset_flip_probability=0.01,
                                before_round_data_depolarization=0.01)

2. Build the detector error model (DEM), and analyze the difference between the two options of circuits.

In [31]:
dem_custom = sc_circuit_custom.detector_error_model(decompose_errors=True)
dem_builtin = sc_circuit_builtin.detector_error_model(decompose_errors=True)

print("Custom circuit error model:")
print(dem_custom)
print(f"Number of errors: {len(dem_custom)}")
print(f"Number of detectors: {dem_custom.num_detectors}")
print(f"Number of observables: {dem_custom.num_observables}")

print("\nBuilt-in circuit error model:")
print(dem_builtin)
print(f"Number of errors: {len(dem_builtin)}")
print(f"Number of detectors: {dem_builtin.num_detectors}")
print(f"Number of observables: {dem_builtin.num_observables}")

Custom circuit error model:
error(0.01262033963927737) D0 D2
error(0.02236793284649182) D0 D4
error(0.01262033963927737) D0 L0
error(0.01262033963927737) D1 D2
error(0.01522666666666665) D1 D3
error(0.005333333333333313) D1 D3 ^ D10
error(0.0006697986788568588) D1 D4
error(0.0006697986788568588) D1 D4 ^ D9
error(0.0006697986788568588) D1 D4 ^ D9 ^ D10
error(0.0006697986788568588) D1 D4 ^ D10
error(0.02236793284649182) D1 D5
error(0.001338700097173321) D1 D5 ^ D9 D10
error(0.001338700097173321) D1 D5 ^ D10
error(0.005995987492949032) D1 D6
error(0.0006697986788568588) D1 D6 ^ D9
error(0.001338700097173321) D1 D6 ^ D9 D10
error(0.0006697986788568588) D1 D6 ^ D9 ^ D10
error(0.002006705456917236) D1 D6 ^ D10
error(0.02682881602833451) D1 L0
error(0.02746267489612923) D2
error(0.004005357180252828) D2 D4
error(0.001338700097173321) D2 D4 ^ D11
error(0.02364674503591481) D2 D6
error(0.001338700097173321) D2 D6 ^ D11
error(0.01262033963927737) D3
error(0.002673815958446298) D3 D5
error(0.0026

It can be seen that the number of detectors differ, as the built-in implementation considers X detectors for Z memory, which is essential for correlation-aware decoding.
Consequently, the built-in methof contains more error mechanisms. Additionally, the coordinates differ in the same method, although that is just a matter of convention.

We now print the two circuits to see the differences clearly.

In [5]:
print("Custom circuit:")
print(sc_circuit_custom)
print("-----------------------------------------------")
print("\nBuilt-in circuit:")
print(sc_circuit_builtin)

Custom circuit:
QUBIT_COORDS(4, 0) 4
QUBIT_COORDS(1, 1) 8
QUBIT_COORDS(3, 1) 10
QUBIT_COORDS(5, 1) 12
QUBIT_COORDS(0, 2) 14
QUBIT_COORDS(2, 2) 16
QUBIT_COORDS(4, 2) 18
QUBIT_COORDS(1, 3) 22
QUBIT_COORDS(3, 3) 24
QUBIT_COORDS(5, 3) 26
QUBIT_COORDS(2, 4) 30
QUBIT_COORDS(4, 4) 32
QUBIT_COORDS(6, 4) 34
QUBIT_COORDS(1, 5) 36
QUBIT_COORDS(3, 5) 38
QUBIT_COORDS(5, 5) 40
QUBIT_COORDS(2, 6) 44
R 8 10 12 22 24 26 36 38 40 4 16 32 44 14 18 30 34
TICK
X_ERROR(0.01) 8 10 12 22 24 26 36 38 40 4 16 32 44 14 18 30 34
TICK
H 4 16 32 44
TICK
DEPOLARIZE1(0.01) 4 16 32 44
TICK
TICK
CX 16 10 32 26 44 38 8 14 12 18 24 30
TICK
DEPOLARIZE2(0.01) 16 10 32 26 44 38 8 14 12 18 24 30
TICK
TICK
CX 16 8 32 24 44 36 22 14 26 18 38 30
TICK
DEPOLARIZE2(0.01) 16 8 32 24 44 36 22 14 26 18 38 30
TICK
TICK
CX 4 12 16 24 32 40 10 18 22 30 26 34
TICK
DEPOLARIZE2(0.01) 4 12 16 24 32 40 10 18 22 30 26 34
TICK
TICK
CX 4 10 16 22 32 38 24 18 36 30 40 34
TICK
DEPOLARIZE2(0.01) 4 10 16 22 32 38 24 18 36 30 40 34
TICK
TICK
H 4 16 

3. Generate a 3D visualization of the decoding graph, including the triggered detectors, the matching edges and other info such that the weights and probabilities.

In [6]:
def plot_mwpm_solution_3d(
    circuit,
    matching,
    syndrome,
    true_obs,
    pred_obs,
    solution_edges,
    show_boundary=False,  # kept for compatibility; boundary is excluded regardless
):
    """
    3D plot of the PyMatching decoding graph:
      - Boundary/aux nodes and their edges are excluded.
      - Detectors are colored by stabilizer type (X vs Z) inferred from (x,y) mod-4 pattern.
      - Fired detectors are highlighted with different colors for fired X vs fired Z.
      - MWPM selected edges are highlighted.
    """

    # --- Fixed colors (not function inputs) ---
    x_color = "#8b180f"        # X detectors (purple)
    z_color = "#063170"        # Z detectors (teal)
    fired_x_color = "#ff3131"  # fired X (orange)
    fired_z_color = "#107ffd"  # fired Z (red)
    edge_color = "#636262"     # non-solution edges
    sol_edge_color = "#8E9100" # MWPM selected edges

    # --- Graph export ---
    G = matching.to_networkx()

    # --- Fired detector set from syndrome ---
    if hasattr(syndrome, "ndim") and syndrome.ndim == 2:
        fired = set(np.where(syndrome[0] == 1)[0].tolist())
    else:
        fired = set(np.where(syndrome == 1)[0].tolist())

    # --- Detector coordinates (real detectors only) ---
    det_xyz_raw = circuit.get_detector_coordinates()  # {det_id: (x,y,t,...)}

    det_xyz = {}
    for k, c in det_xyz_raw.items():
        try:
            ki = int(k)
        except Exception:
            continue
        c = list(c)
        while len(c) < 3:
            c.append(0.0)
        det_xyz[ki] = (float(c[0]), float(c[1]), float(c[2]))

    # --- Keep only detector nodes that have coords (drop boundary/aux nodes) ---
    node_int = {}
    for n in G.nodes():
        try:
            node_int[n] = int(n)
        except Exception:
            node_int[n] = None

    det_nodes = [n for n in G.nodes() if node_int.get(n, None) in det_xyz]
    det_node_set = set(det_nodes)

    # positions for plotting (only detector nodes)
    pos = {n: det_xyz[node_int[n]] for n in det_nodes}

    # filtered edge list: both endpoints are detector nodes
    det_edges = []
    for u, v, data in G.edges(data=True):
        if u in det_node_set and v in det_node_set:
            det_edges.append((u, v, data))

    # --- MWPM solution edges: normalize to detector ids, then map back to graph nodes ---
    sol_set_int = {tuple(sorted((int(e[0]), int(e[1])))) for e in solution_edges}

    int_to_node = {}
    for n in det_nodes:
        int_to_node[node_int[n]] = n

    sol_set_nodes = set()
    for a, b in sol_set_int:
        if a in int_to_node and b in int_to_node:
            na, nb = int_to_node[a], int_to_node[b]
            sol_set_nodes.add(tuple(sorted((na, nb), key=lambda x: node_int[x])))

    # --- Infer detector type (X vs Z) using your lattice mod-4 pattern ---
    # Your stabilizer placement rules:
    #   X: (i%4==0 and j%4==0) or (i%4==2 and j%4==2)
    #   Z: (i%4==0 and j%4==2) or (i%4==2 and j%4==0)
    # In DETECTOR coords you use [j, i, round] => x=j, y=i
    def infer_stab_type(n):
        x, y, _ = pos[n]
        j = int(round(x))
        i = int(round(y))
        if (i % 4 == 0 and j % 4 == 0) or (i % 4 == 2 and j % 4 == 2):
            return "X"
        if (i % 4 == 0 and j % 4 == 2) or (i % 4 == 2 and j % 4 == 0):
            return "Z"
        # Fallback (shouldn't happen often): separate by parity instead of lumping everything
        return "X" if ((i + j) % 2 == 0) else "Z"

    # --- Build edge traces ---
    all_x, all_y, all_z = [], [], []
    sel_x, sel_y, sel_z = [], [], []
    mid_x, mid_y, mid_z, mid_txt = [], [], [], []

    for u, v, data in det_edges:
        x0, y0, z0 = pos[u]
        x1, y1, z1 = pos[v]

        w = data.get("weight", None)
        p = data.get("error_probability", data.get("p", None))

        mid_x.append((x0 + x1) / 2)
        mid_y.append((y0 + y1) / 2)
        mid_z.append((z0 + z1) / 2)
        mid_txt.append(f"{node_int[u]}—{node_int[v]}<br>weight={w}<br>p={p}")

        e_nodes = tuple(sorted((u, v), key=lambda x: node_int[x]))
        if e_nodes in sol_set_nodes:
            sel_x += [x0, x1, None]
            sel_y += [y0, y1, None]
            sel_z += [z0, z1, None]
        else:
            all_x += [x0, x1, None]
            all_y += [y0, y1, None]
            all_z += [z0, z1, None]

    # --- Build node traces (X, Z, fired X, fired Z) ---
    x_x, x_y, x_z, x_txt = [], [], [], []
    z_x, z_y, z_z, z_txt = [], [], [], []
    fx_x, fx_y, fx_z, fx_txt = [], [], [], []
    fz_x, fz_y, fz_z, fz_txt = [], [], [], []

    for n in det_nodes:
        x, y, z = pos[n]
        t = infer_stab_type(n)
        det_id = node_int[n]
        label = f"D{det_id} ({t})"

        if det_id in fired:
            if t == "X":
                fx_x.append(x); fx_y.append(y); fx_z.append(z)
                fx_txt.append(label + " • fired")
            else:
                fz_x.append(x); fz_y.append(y); fz_z.append(z)
                fz_txt.append(label + " • fired")
        else:
            if t == "X":
                x_x.append(x); x_y.append(y); x_z.append(z); x_txt.append(label)
            else:
                z_x.append(x); z_y.append(y); z_z.append(z); z_txt.append(label)

    # --- Plot ---
    fig = go.Figure()

    fig.add_trace(go.Scatter3d(
        x=all_x, y=all_y, z=all_z,
        mode="lines",
        line=dict(width=2, color=edge_color),
        name="graph edges",
        hoverinfo="skip",
    ))

    fig.add_trace(go.Scatter3d(
        x=sel_x, y=sel_y, z=sel_z,
        mode="lines",
        line=dict(width=7, color=sol_edge_color),
        name="MWPM selected",
        hoverinfo="skip",
    ))

    fig.add_trace(go.Scatter3d(
        x=mid_x, y=mid_y, z=mid_z,
        mode="markers",
        marker=dict(size=2, opacity=0.0),
        text=mid_txt,
        hoverinfo="text",
        name="edge info (hover)",
    ))

    fig.add_trace(go.Scatter3d(
        x=x_x, y=x_y, z=x_z,
        mode="markers",
        marker=dict(size=4, color=x_color),
        text=x_txt,
        hoverinfo="text",
        name="X detectors",
    ))

    fig.add_trace(go.Scatter3d(
        x=z_x, y=z_y, z=z_z,
        mode="markers",
        marker=dict(size=4, color=z_color),
        text=z_txt,
        hoverinfo="text",
        name="Z detectors",
    ))

    fig.add_trace(go.Scatter3d(
        x=fx_x, y=fx_y, z=fx_z,
        mode="markers",
        marker=dict(size=8, color=fired_x_color),
        text=fx_txt,
        hoverinfo="text",
        name="fired X",
    ))

    fig.add_trace(go.Scatter3d(
        x=fz_x, y=fz_y, z=fz_z,
        mode="markers",
        marker=dict(size=8, color=fired_z_color),
        text=fz_txt,
        hoverinfo="text",
        name="fired Z",
    ))

    # title-safe extraction
    try:
        p_obs = pred_obs[0][0] if hasattr(pred_obs, "ndim") and pred_obs.ndim > 1 else pred_obs[0]
    except Exception:
        p_obs = pred_obs

    fig.update_layout(
        title=f"MWPM (True Obs={1 if bool(true_obs) else 0}, Pred Obs={p_obs})",
        scene=dict(
            xaxis_title="x",
            yaxis_title="y",
            zaxis_title="t (round)",
            aspectmode="data",
        ),
        legend=dict(itemsizing="constant"),
    )

    fig.show()
    return G, fig


4. Define the decoding graph, including the weights derived from either the Stim circuit ot DEM.

In [7]:
enable_correlations = False # Set to True to include correlated errors in the matching graph, False for independent errors only

matching = pymatching.Matching.from_stim_circuit(sc_circuit_builtin, enable_correlations=enable_correlations)
decoding_weights = matching.edges()
print("Number of edges in the decoding graph:", len(decoding_weights))
print("\nAll edges with weights and probabilities:")
for u, v, data in decoding_weights:
    print(f"Edge ({u}, {v}): weight={data.get('weight', None)}, p={data.get('error_probability', None)}")

Number of edges in the decoding graph: 78

All edges with weights and probabilities:
Edge (0, None): weight=3.8089606069523247, p=0.021690311111111034
Edge (0, 1): weight=3.9377826673290692, p=0.01911873511075362
Edge (0, 8): weight=3.7775050259622516, p=0.02236793284649182
Edge (1, 2): weight=3.9377826673290692, p=0.01911873511075362
Edge (1, 5): weight=3.666761033525035, p=0.024922133333333308
Edge (4, None): weight=2.5547187084193, p=0.0721101172219637
Edge (1, 8): weight=5.228431239083874, p=0.005333333333333312
Edge (1, None): weight=2.9000941400494242, p=0.052148909595990396
Edge (2, None): weight=2.9000941400494247, p=0.05214890959599036
Edge (2, 3): weight=3.592483713972072, p=0.026792281125925837
Edge (6, None): weight=2.247765744704371, p=0.09554236148440248
Edge (2, 5): weight=4.535312808407135, p=0.010609777777777737
Edge (6, 9): weight=3.781279583139722, p=0.022285540946172723
Edge (9, None): weight=2.247765744704372, p=0.09554236148440248
Edge (2, 8): weight=5.92157123199

5. We now sample syndromes derived from the Stim circuit, although maybe in the future we sample from the detector error model. I believe  the former is more accurate while the latter is faster.

In [8]:
def extract_syndrome_from_circuit(circuit, n_shots=1, seed=None):
    if seed is not None:
        sampler = circuit.compile_detector_sampler(seed=seed)
    else:
        sampler = circuit.compile_detector_sampler()

    syndrome, true_obs = sampler.sample(shots=n_shots, separate_observables=True)

    return syndrome, true_obs

In [9]:
syndrome, true_obs = extract_syndrome_from_circuit(sc_circuit_builtin, n_shots=1)
print("Syndrome:", syndrome)

pred_obs = matching.decode_batch(syndrome)
solution_edges = set(map(tuple, matching.decode_to_edges_array(syndrome).tolist()))
#print(solution_edges)

G, fig = plot_mwpm_solution_3d(sc_circuit_builtin, matching, syndrome, true_obs, pred_obs, solution_edges)


Syndrome: [[False False False False  True  True False False False False  True False
   True False False False False False  True False False False False False]]


6. Print the weights as obtained directly from the Stim circuit/DEM.

In [10]:
for u, v, data in decoding_weights:
    u = -1 if u is None else u
    v = -1 if v is None else v
    if (u, v) in solution_edges or (v, u) in solution_edges:
        print(f"Selected edge: ({u}, {v}), weight={data.get('weight', None)}, p={data.get('error_probability', None)}")

Selected edge: (5, -1), weight=2.877782054133115, p=0.05326286763777633
Selected edge: (4, 12), weight=3.316191729963172, p=0.035019875603091326
Selected edge: (10, 18), weight=3.6667610335250345, p=0.024922133333333308


7. Reweight the edges of the decoding graph

In [11]:
u, v = next(iter(solution_edges))
if v == -1:
    edge_data = matching.get_boundary_edge_data(u)
else:
    edge_data = matching.get_edge_data(u, v)
    
print(f"\nModifying weight of selected edge ({u}, {v}) to simulate a change in error model...")
new_weight = edge_data["weight"] - np.log(1.2)

if v == -1:
    matching.add_boundary_edge(
    u, 
    weight=new_weight, 
    error_probability=1/(1 + np.exp(new_weight)), 
    merge_strategy="replace"
)
else:
    matching.add_edge(
    u, 
    v, 
    weight=new_weight, 
    error_probability=1/(1 + np.exp(new_weight)), 
    merge_strategy="replace"
)

print("\nNew decoding weights for each edge:\n")
new_decoding_weights = matching.edges()
for u, v, data in new_decoding_weights:
    u = int(u) if u is not None else -1
    v = int(v) if v is not None else -1
    if (u, v) in solution_edges or (v, u) in solution_edges:
        print(f"Selected edge: ({u}, {v}), weight={data.get('weight', None)}, p={data.get('error_probability', None)}")




Modifying weight of selected edge (5, -1) to simulate a change in error model...

New decoding weights for each edge:

Selected edge: (5, -1), weight=2.6954604973391607, p=0.06324175373367215
Selected edge: (4, 12), weight=3.316191729963172, p=0.035019875603091326
Selected edge: (10, 18), weight=3.6667610335250345, p=0.024922133333333308
